# Пара 3+. OpenAlex API: библиография вокруг Бунина

Бесплатный каталог научных работ. Ключ желателен, но не обязателен: https://openalex.org/settings/api

**Результат ячейки:** файл `openalex_bunin_raw.txt` в файлах Colab (слева) — скачайте его.


In [ ]:
!pip -q install requests


In [ ]:
import time
from pathlib import Path
import requests

API = "https://api.openalex.org/works"
API_KEY = ""  # вставьте ключ или оставьте пустым
OUT = Path("openalex_bunin_raw.txt")

QUERIES = [
    {"tag": "A_bunin", "mode": "search", "value": '"Ivan Bunin"', "title_ru": "Бунин как объект изучения"},
    {"tag": "B_dark_avenues", "mode": "filter", "value": "title.search:Dark Avenues", "title_ru": "Dark Avenues / Тёмные аллеи"},
    {"tag": "C_emigre", "mode": "search", "value": '"Russian emigre literature" OR "Russian émigré literature"', "title_ru": "Эмигрантская литература"},
]

def fetch(mode, value, per_page=15):
    params = {"per-page": per_page, "select": "id,doi,display_name,publication_year,cited_by_count,authorships,primary_topic,type"}
    if mode == "search":
        params["search"] = value
    else:
        params["filter"] = value
    if API_KEY:
        params["api_key"] = API_KEY
    r = requests.get(API, params=params, timeout=60)
    r.raise_for_status()
    return r.json()

lines = ["OpenAlex raw dump for Bunin labs", ""]
for item in QUERIES:
    data = fetch(item["mode"], item["value"])
    print(item["tag"], "count=", data["meta"]["count"], "got", len(data["results"]))
    lines += ["="*60, item["tag"], item["title_ru"], item["value"], f"meta.count={data['meta']['count']}", "-"*60]
    for i, w in enumerate(data["results"], 1):
        authors = "; ".join(a["author"]["display_name"] for a in (w.get("authorships") or []))
        topic = ((w.get("primary_topic") or {}).get("display_name")) or "—"
        lines.append(f"{i}. {w.get('publication_year')} | {w.get('display_name')}\n   {authors}\n   topic: {topic}\n")
    time.sleep(0.2)

OUT.write_text("\n".join(lines), encoding="utf-8")
print("saved", OUT.resolve())
print(OUT.read_text(encoding="utf-8")[:800])


### Задание если успели быстро
Вычеркните 3 шумные записи и оставьте 5 полезных для филолога. Напишите одной фразой: зачем эта подборка перед анализом «Тёмных аллей».
